# Advanced Problems with Solutions: Custom Classes and Hashing

This notebook develops advanced hashing problems in a **tutorial style**. Rather than presenting a finished class immediately, each problem is broken into small logical steps:

1. state the desired behavior,
2. inspect a first attempt,
3. predict or observe the failure,
4. explain the reason,
5. repair the design,
6. verify the result with focused tests.

The examples use different domains from the source lesson so that the reasoning—not memorization of a particular class—does the work.

## What we will practice

By the end of the notebook, we will have worked with:

- identity equality versus value equality,
- the `__eq__` / `__hash__` contract,
- `NotImplemented` and reflected comparisons,
- canonicalization before hashing,
- order-independent keys,
- mutable descriptive data that is excluded from identity,
- shallow versus deep immutability,
- dataclass hash-generation rules,
- inheritance traps,
- carefully designed cross-type equality,
- safe and unsafe hash caching,
- collision diagnostics and performance,
- runtime hashes versus persistent digests,
- recursive freezing for memoization keys,
- immutable snapshot keys for mutable domain objects,
- reusable contract tests.

## The rule that controls the entire notebook

A hashable class must obey this implication:

```text
if a == b, then hash(a) == hash(b)
```

The reverse is **not** required. Two unequal objects may have the same hash; that situation is called a collision.

A second practical rule is just as important:

> Every piece of state that affects equality must remain stable while the object is being used as a dictionary key or set member.

The object does not have to be deeply immutable in every possible sense. What must remain stable is its **logical identity**.

## Notebook setup

The helper below lets us demonstrate expected failures without stopping execution of the notebook. This is useful in a tutorial because some incorrect designs are worth running and inspecting.

In [1]:
from __future__ import annotations

from collections.abc import Hashable, Mapping, Set as AbstractSet
from dataclasses import dataclass, field
from datetime import date
from hashlib import sha256
from itertools import combinations
from pathlib import Path
from timeit import timeit
from typing import Any, ClassVar
import json
import os
import subprocess
import sys
import unicodedata


def show_exception(label, operation):
    """Run operation and print a compact representation of an expected error."""
    try:
        result = operation()
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(f"{label}: {result!r}")


def show_hashability(obj):
    """Display whether an object satisfies the Hashable protocol at runtime."""
    print(f"type={type(obj).__name__:<24} Hashable={isinstance(obj, Hashable)!s:<5}", end="")
    try:
        print(f" hash={hash(obj)}")
    except TypeError as exc:
        print(f" hash error={exc}")

---

# Problem 1 — Repair a value object in stages

We want a `ProductCode` to act like a value. Two independently created codes with the same department and number should retrieve the same dictionary entry.

We will deliberately begin with the default object behavior.

### Step 1: Start with a class that has no custom equality

In [2]:
class ProductCode:
    def __init__(self, department, number):
        self.department = department
        self.number = number

    def __repr__(self):
        return f"ProductCode({self.department!r}, {self.number!r})"


code_1 = ProductCode("BOOK", 104)
code_2 = ProductCode("BOOK", 104)

print(code_1 is code_2)
print(code_1 == code_2)
print(hash(code_1) == hash(code_2))

False
False
False


The two instances contain the same values, but the default equality inherited from `object` is identity-based. Therefore `code_1` and `code_2` are different keys.

In [3]:
inventory = {code_1: 27}
print(inventory.get(code_2, "not found"))

not found


### Step 2: Add value equality

In [4]:
class ProductCode:
    def __init__(self, department, number):
        self.department = department
        self.number = number

    def __repr__(self):
        return f"ProductCode({self.department!r}, {self.number!r})"

    def __eq__(self, other):
        if isinstance(other, ProductCode):
            return (
                self.department == other.department
                and self.number == other.number
            )
        return NotImplemented


code_1 = ProductCode("BOOK", 104)
code_2 = ProductCode("BOOK", 104)
print(code_1 == code_2)
show_hashability(code_1)

True
type=ProductCode              Hashable=False hash error=unhashable type: 'ProductCode'


Once a class defines `__eq__`, Python cannot safely keep the old identity-based hash. Equal objects could then have different hashes. Python therefore sets `ProductCode.__hash__` to `None` unless we explicitly provide a compatible implementation.

In [5]:
print(ProductCode.__hash__)
show_exception("dictionary construction", lambda: {code_1: 27})

None
dictionary construction: TypeError: unhashable type: 'ProductCode'


### Step 3: Hash exactly the state used by equality

In [6]:
class ProductCode:
    def __init__(self, department, number):
        self.department = department
        self.number = number

    def __repr__(self):
        return f"ProductCode({self.department!r}, {self.number!r})"

    def __eq__(self, other):
        if isinstance(other, ProductCode):
            return (
                self.department == other.department
                and self.number == other.number
            )
        return NotImplemented

    def __hash__(self):
        return hash((self.department, self.number))


code_1 = ProductCode("BOOK", 104)
code_2 = ProductCode("BOOK", 104)

inventory = {code_1: 27}
print(code_1 == code_2)
print(hash(code_1) == hash(code_2))
print(inventory[code_2])

True
True
27


### Solution takeaway

The equality fields and hash fields must describe the same logical identity. Hashing the tuple of those fields is usually clearer and safer than inventing an arithmetic formula.

---

# Problem 2 — Why returning `NotImplemented` matters

A comparison method sometimes receives an object of an unrelated type. Returning `False` immediately may seem harmless, but it prevents Python from asking the other operand whether it knows how to perform the comparison.

### Step 1: Construct an asymmetric comparison

In [7]:
class LegacyCode:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        if hasattr(other, "value"):
            return self.value == other.value
        return NotImplemented

    def __hash__(self):
        return hash(self.value)


class BadModernCode:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        if isinstance(other, BadModernCode):
            return self.value == other.value
        return False  # closes the door too early

    def __hash__(self):
        return hash(self.value)


legacy = LegacyCode("A-19")
modern = BadModernCode("A-19")
print(modern == legacy)
print(legacy == modern)

False
True


The first expression calls `BadModernCode.__eq__`, which returns the final answer `False`. Python does not try `LegacyCode.__eq__` afterward.

The second expression starts with `LegacyCode.__eq__`, which understands objects that expose a compatible `value` attribute, so it returns `True`.

Equality should not depend on the operand order like this.

### Step 2: Let Python try the reflected comparison

In [8]:
class ModernCode:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        if isinstance(other, ModernCode):
            return self.value == other.value
        return NotImplemented

    def __hash__(self):
        return hash(self.value)


legacy = LegacyCode("A-19")
modern = ModernCode("A-19")
print(modern == legacy)
print(legacy == modern)
print(hash(modern) == hash(legacy))

True
True
True


### Solution takeaway

Use `NotImplemented` when your class does not know how to compare with the other type. It is a protocol signal, not an error and not the same thing as returning `False`.

When two different hashable types are allowed to compare equal, their hash implementations must also be coordinated.

---

# Problem 3 — Canonicalize before equality and hashing

Suppose tags are considered equal after:

- Unicode normalization,
- trimming surrounding whitespace,
- collapsing internal whitespace,
- case-insensitive comparison.

The canonical form should be computed once and then used by both `__eq__` and `__hash__`.

### Step 1: Observe why plain lowercase is not enough

In [9]:
print("Straße".lower())
print("STRASSE".lower())
print("Straße".casefold())
print("STRASSE".casefold())

straße
strasse
strasse
strasse


`casefold()` is designed for caseless matching and handles more Unicode cases than `lower()`.

### Step 2: Build a single normalization function

In [10]:
def normalize_tag(text):
    normalized = unicodedata.normalize("NFKC", text)
    collapsed = " ".join(normalized.split())
    return collapsed.casefold()


examples = [
    "  Data   Science ",
    "DATA SCIENCE",
    "Straße",
    "STRASSE",
]

for value in examples:
    print(f"{value!r:22} -> {normalize_tag(value)!r}")

'  Data   Science '    -> 'data science'
'DATA SCIENCE'         -> 'data science'
'Straße'               -> 'strasse'
'STRASSE'              -> 'strasse'


### Step 3: Store the canonical identity, not the raw spelling

In [11]:
@dataclass(frozen=True)
class TagKey:
    original: str = field(compare=False, hash=False)
    canonical: str = field(init=False)

    def __post_init__(self):
        object.__setattr__(self, "canonical", normalize_tag(self.original))

    def __str__(self):
        return self.original


left = TagKey("  Straße ")
right = TagKey("STRASSE")

print(left.canonical)
print(right.canonical)
print(left == right)
print(hash(left) == hash(right))

strasse
strasse
True
True


In [12]:
article_counts = {TagKey("Data   Science"): 12}
print(article_counts[TagKey(" data science ")])

12


### Solution takeaway

Do not normalize independently in several methods. A single canonical representation prevents equality and hashing from drifting apart. The original spelling can still be retained for display, provided it is excluded from identity.

---

# Problem 4 — Hash an unordered pair correctly

An undirected graph edge from `A` to `B` is the same edge as one from `B` to `A`.

A common mistake is to make equality order-independent but leave the hash order-dependent.

### Step 1: Inspect the broken design

In [13]:
class BadUndirectedEdge:
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def __repr__(self):
        return f"BadUndirectedEdge({self.left!r}, {self.right!r})"

    def __eq__(self, other):
        if isinstance(other, BadUndirectedEdge):
            return {self.left, self.right} == {other.left, other.right}
        return NotImplemented

    def __hash__(self):
        return hash((self.left, self.right))  # order-dependent


edge_1 = BadUndirectedEdge("A", "B")
edge_2 = BadUndirectedEdge("B", "A")

print(edge_1 == edge_2)
print(hash(edge_1) == hash(edge_2))

True
False


The objects compare equal, yet their hashes differ. That directly violates the hash contract.

In [14]:
broken_edges = {edge_1: "first", edge_2: "second"}
print(len(broken_edges))
print(broken_edges)

2
{BadUndirectedEdge('A', 'B'): 'first', BadUndirectedEdge('B', 'A'): 'second'}


Because the hashes sent the equal keys to different search regions, the dictionary retained two entries that the class itself claims are equal.

### Step 2: Represent the identity as a `frozenset`

In [15]:
@dataclass(frozen=True)
class UndirectedEdge:
    endpoints: frozenset

    def __init__(self, left, right):
        object.__setattr__(self, "endpoints", frozenset((left, right)))

    def __repr__(self):
        values = tuple(self.endpoints)
        return f"UndirectedEdge({values!r})"


edge_1 = UndirectedEdge("A", "B")
edge_2 = UndirectedEdge("B", "A")

print(edge_1 == edge_2)
print(hash(edge_1) == hash(edge_2))
print(len({edge_1: "first", edge_2: "second"}))

True
True
1


### Step 3: Check the self-loop case

A self-loop uses one distinct endpoint. That is still different from an edge connecting two distinct nodes.

In [16]:
loop = UndirectedEdge("A", "A")
regular = UndirectedEdge("A", "B")
print(loop)
print(loop == regular)
print(len({loop, regular}))

UndirectedEdge(('A',))
False
2


### Solution takeaway

Choose a hashable identity representation whose own equality already matches the domain rule. For an unordered set of endpoints, a `frozenset` is a natural representation.

---

# Problem 5 — Separate identity from mutable descriptive data

A log event is identified by `(source, event_id)`. It also carries mutable metadata used for display and enrichment.

The metadata must not affect whether two event keys are considered the same.

### Step 1: Declare identity fields explicitly

In [17]:
@dataclass(frozen=True)
class EventKey:
    source: str
    event_id: int
    metadata: dict[str, Any] = field(
        default_factory=dict,
        compare=False,
        hash=False,
        repr=False,
    )


first = EventKey("billing", 501, {"severity": "warning"})
second = EventKey("billing", 501, {"severity": "critical", "retry": 2})

print(first == second)
print(hash(first) == hash(second))
print(len({first, second}))

True
True
1


Although the metadata dictionaries differ, the objects have the same logical identity. Dataclass fields marked `compare=False` do not participate in generated equality. Marking the same field `hash=False` makes that decision explicit for hashing as well.

### Step 2: Mutate nested metadata

In [18]:
events = {first: "stored event"}
old_hash = hash(first)

first.metadata["acknowledged"] = True

print(first.metadata)
print(hash(first) == old_hash)
print(events[first])
print(events[EventKey("billing", 501)])

{'severity': 'warning', 'acknowledged': True}
True
stored event
stored event


A frozen dataclass prevents rebinding `first.metadata`, but it does not recursively freeze the dictionary stored inside it. This nested mutation is safe **for dictionary membership** only because metadata affects neither equality nor hashing.

### Solution takeaway

A hashable object may contain mutable descriptive state, but that state must be excluded from logical identity. Document this distinction clearly because later changes to `compare` or `hash` settings can silently break the design.

---

# Problem 6 — Diagnose a dictionary key that disappears after mutation

We will create a mutable route key whose origin and destination participate in equality and hashing. Then we will mutate one of those fields after insertion.

### Step 1: Insert the key while it has its original state

In [19]:
class MutableRouteKey:
    def __init__(self, origin, destination):
        self.origin = origin
        self.destination = destination

    def __repr__(self):
        return f"MutableRouteKey({self.origin!r}, {self.destination!r})"

    def __eq__(self, other):
        if isinstance(other, MutableRouteKey):
            return (
                self.origin == other.origin
                and self.destination == other.destination
            )
        return NotImplemented

    def __hash__(self):
        return hash((self.origin, self.destination))


route = MutableRouteKey("SOF", "LHR")
fares = {route: 189}

print(hash(route))
print(fares[MutableRouteKey("SOF", "LHR")])

4223610348832893474
189


### Step 2: Mutate a hash-relevant field

In [20]:
old_hash = hash(route)
route.destination = "CDG"
new_hash = hash(route)

print(old_hash)
print(new_hash)
print(old_hash == new_hash)

4223610348832893474
1761095945872812438
False


In [21]:
show_exception("lookup with the mutated object", lambda: fares[route])
show_exception(
    "lookup with a fresh equal object",
    lambda: fares[MutableRouteKey("SOF", "CDG")],
)

lookup with the mutated object: KeyError: MutableRouteKey('SOF', 'CDG')
lookup with a fresh equal object: KeyError: MutableRouteKey('SOF', 'CDG')


The entry is still physically present, but it is stored under the hash computed from the old state. Looking it up now starts from the hash of the new state.

In [22]:
for stored_key, value in fares.items():
    print(stored_key, value)

MutableRouteKey('SOF', 'CDG') 189


### Step 3: Replace the mutable key with an immutable value object

In [23]:
@dataclass(frozen=True)
class RouteKey:
    origin: str
    destination: str


route = RouteKey("SOF", "LHR")
fares = {route: 189}

print(fares[RouteKey("SOF", "LHR")])
show_exception(
    "attempted mutation",
    lambda: setattr(route, "destination", "CDG"),
)

189
attempted mutation: FrozenInstanceError: cannot assign to field 'destination'


### Solution takeaway

Once an object becomes a key, its equality and hash identity must not change. Prefer immutable value objects for keys rather than relying on every caller to remember a mutation rule.

---

# Problem 7 — Frozen does not mean deeply hashable

A frozen dataclass can still contain an unhashable field. Freezing blocks attribute assignment; it does not automatically convert a list into an immutable collection.

### Step 1: Freeze a class that contains a list

In [24]:
@dataclass(frozen=True)
class BadBatchKey:
    warehouse: str
    item_ids: list[int]


bad_batch = BadBatchKey("SOF-1", [10, 20, 30])
print(bad_batch)
show_hashability(bad_batch)

BadBatchKey(warehouse='SOF-1', item_ids=[10, 20, 30])
type=BadBatchKey              Hashable=True  hash error=unhashable type: 'list'


The generated hash attempts to hash every identity field. The list is unhashable, so the entire object is unhashable even though the dataclass is frozen.

### Step 2: Canonicalize the incoming collection to a tuple

In [25]:
@dataclass(frozen=True)
class BatchKey:
    warehouse: str
    item_ids: tuple[int, ...]

    def __init__(self, warehouse, item_ids):
        object.__setattr__(self, "warehouse", warehouse)
        object.__setattr__(self, "item_ids", tuple(item_ids))


source_ids = [10, 20, 30]
batch = BatchKey("SOF-1", source_ids)

print(batch)
show_hashability(batch)

BatchKey(warehouse='SOF-1', item_ids=(10, 20, 30))
type=BatchKey                 Hashable=True  hash=-9167844534474824034


### Step 3: Verify that later source-list mutation does not leak into the key

In [26]:
batches = {batch: "ready"}
source_ids.append(40)

print(source_ids)
print(batch.item_ids)
print(batches[BatchKey("SOF-1", [10, 20, 30])])

[10, 20, 30, 40]
(10, 20, 30)
ready


### Solution takeaway

Immutability should be enforced at the boundary. Convert mutable input collections into immutable internal representations during construction rather than merely documenting that callers should not modify them.

---

# Problem 8 — Understand the dataclass equality/hash matrix

Dataclasses generate or suppress `__hash__` according to several options. We will compare four common configurations.

### Step 1: Define the four configurations

In [27]:
@dataclass(eq=False)
class IdentityRecord:
    value: int


@dataclass
class MutableValueRecord:
    value: int


@dataclass(frozen=True)
class FrozenValueRecord:
    value: int


@dataclass(unsafe_hash=True)
class UnsafeValueRecord:
    value: int

### Step 2: Inspect equality and hashability

In [28]:
records = [
    IdentityRecord(7),
    MutableValueRecord(7),
    FrozenValueRecord(7),
    UnsafeValueRecord(7),
]

for record in records:
    twin = type(record)(7)
    print(f"\n{type(record).__name__}")
    print("equal to twin:", record == twin)
    print("class __hash__:", type(record).__hash__)
    show_hashability(record)


IdentityRecord
equal to twin: False
class __hash__: <slot wrapper '__hash__' of 'object' objects>
type=IdentityRecord           Hashable=True  hash=115739300433

MutableValueRecord
equal to twin: True
class __hash__: None
type=MutableValueRecord       Hashable=False hash error=unhashable type: 'MutableValueRecord'

FrozenValueRecord
equal to twin: True
class __hash__: <function FrozenValueRecord.__hash__ at 0x000001AF299DB560>
type=FrozenValueRecord        Hashable=True  hash=-6198871656470064846

UnsafeValueRecord
equal to twin: True
class __hash__: <function UnsafeValueRecord.__hash__ at 0x000001AF299DB920>
type=UnsafeValueRecord        Hashable=True  hash=-6198871656470064846


The usual behavior is:

- `eq=False`: identity equality and the inherited identity hash remain.
- `eq=True, frozen=False`: value equality is generated, so hashing is disabled.
- `eq=True, frozen=True`: value equality and a compatible value hash are generated.
- `unsafe_hash=True`: a value hash is generated even though fields may still mutate.

### Step 3: Demonstrate why the option is called `unsafe_hash`

In [29]:
unsafe = UnsafeValueRecord(7)
lookup = {unsafe: "stored"}
print(lookup[UnsafeValueRecord(7)])

unsafe.value = 8
show_exception("lookup after mutation", lambda: lookup[unsafe])

stored
lookup after mutation: KeyError: UnsafeValueRecord(value=8)


### Solution takeaway

`unsafe_hash=True` is not an immutability feature. It should be reserved for cases where external logic truly guarantees that equality-relevant fields will not change while the object is hashed.

---

# Problem 9 — Avoid an inheritance equality/hash trap

A base key identifies a user by `user_id`. A subclass adds a tenant and changes the hash to include that tenant, but inherits the base equality unchanged.

This creates equal objects with different hashes.

### Step 1: Build the inconsistent hierarchy

In [30]:
class UserKey:
    def __init__(self, user_id):
        self.user_id = user_id

    def __eq__(self, other):
        if isinstance(other, UserKey):
            return self.user_id == other.user_id
        return NotImplemented

    def __hash__(self):
        return hash(self.user_id)

    def __repr__(self):
        return f"UserKey({self.user_id!r})"


class TenantUserKey(UserKey):
    def __init__(self, tenant, user_id):
        super().__init__(user_id)
        self.tenant = tenant

    def __hash__(self):
        return hash((self.tenant, self.user_id))

    def __repr__(self):
        return f"TenantUserKey({self.tenant!r}, {self.user_id!r})"


base = UserKey(42)
tenanted = TenantUserKey("eu", 42)

print(base == tenanted)
print(tenanted == base)
print(hash(base) == hash(tenanted))

True
True
False


In [31]:
users = {base: "base", tenanted: "tenant"}
print(len(users))
print(users)

2
{UserKey(42): 'base', TenantUserKey('eu', 42): 'tenant'}


The inherited equality says the objects are equal because their `user_id` values match. The subclass hash says tenant is part of identity. The two methods therefore disagree about what the object means.

### Step 2: Use exact-type equality when subclasses represent different key domains

In [32]:
class StrictUserKey:
    def __init__(self, user_id):
        self.user_id = user_id

    def __eq__(self, other):
        if type(other) is type(self):
            return self.user_id == other.user_id
        return NotImplemented

    def __hash__(self):
        return hash((type(self), self.user_id))


class StrictTenantUserKey(StrictUserKey):
    def __init__(self, tenant, user_id):
        super().__init__(user_id)
        self.tenant = tenant

    def __eq__(self, other):
        if type(other) is type(self):
            return (
                self.tenant == other.tenant
                and self.user_id == other.user_id
            )
        return NotImplemented

    def __hash__(self):
        return hash((type(self), self.tenant, self.user_id))


base = StrictUserKey(42)
tenanted = StrictTenantUserKey("eu", 42)

print(base == tenanted)
print(tenanted == base)
print(len({base, tenanted}))

False
False
2


### Solution takeaway

Before using `isinstance` inside value equality, decide whether subclasses genuinely belong to the same equality domain. If a subclass adds identity fields, exact-type equality is often safer.

---

# Problem 10 — Design cross-type equality only where hashes can align

A `DateKey` wrapper should be interchangeable with `datetime.date` in dictionary lookups. It may also accept an ISO string at construction time, but that does **not** mean it should compare equal to a string.

### Step 1: Use parsing at the constructor boundary

In [33]:
@dataclass(frozen=True, eq=False)
class DateKey:
    _value: date = field(repr=False)

    def __init__(self, value):
        if isinstance(value, str):
            value = date.fromisoformat(value)
        if not isinstance(value, date):
            raise TypeError("DateKey requires a date or ISO date string")
        object.__setattr__(self, "_value", value)

    @property
    def value(self):
        return self._value

    def __repr__(self):
        return f"DateKey({self._value.isoformat()!r})"

    def __eq__(self, other):
        if isinstance(other, DateKey):
            return self._value == other._value
        if isinstance(other, date):
            return self._value == other
        return NotImplemented

    def __hash__(self):
        return hash(self._value)


wrapped = DateKey("2026-08-04")
plain = date(2026, 8, 4)

print(wrapped == plain)
print(plain == wrapped)
print(hash(wrapped) == hash(plain))

True
True
True


In [34]:
schedule = {wrapped: "release"}
print(schedule[plain])
print(schedule[DateKey(plain)])

release
release


### Step 2: Explain why equality with the source string is intentionally absent

In [35]:
iso_text = "2026-08-04"
print(wrapped == iso_text)
print(hash(plain))
print(hash(iso_text))

False
697963647844965196
4095186493067902625


If `DateKey("2026-08-04")` compared equal to both the `date` object and the string, then its one hash value would need to equal both `hash(date(...))` and `hash("2026-08-04")`. Those external types do not promise equal hashes.

Parsing compatibility and equality compatibility are different design decisions.

### Solution takeaway

Cross-type equality is safe only when all participating types share the same equality meaning and compatible hash behavior. Otherwise, convert at construction time and keep equality inside one coherent value domain.

---

# Problem 11 — A cached hash does not repair mutable identity

A developer may notice that mutation changes a key's hash and try to cache the first hash value. That prevents the numeric hash from changing, but it does not preserve the equality/hash contract.

### Step 1: Cache a hash on a mutable object

In [36]:
class BadCachedKey:
    def __init__(self, value):
        self.value = value
        self._cached_hash = hash(value)

    def __eq__(self, other):
        if isinstance(other, BadCachedKey):
            return self.value == other.value
        return NotImplemented

    def __hash__(self):
        return self._cached_hash


original = BadCachedKey("alpha")
lookup = {original: "stored"}
original.value = "beta"
fresh = BadCachedKey("beta")

print(original == fresh)
print(hash(original))
print(hash(fresh))
print(hash(original) == hash(fresh))

True
3723529817686220257
8260063330434956831
False


The mutated object now compares equal to a fresh `"beta"` key, but it still returns the cached hash of `"alpha"`. The cache preserved stability by breaking compatibility with equality.

### Step 2: Cache only after making identity immutable

In [37]:
class ImmutablePathKey:
    __slots__ = ("_parts", "_hash")

    hash_computations: ClassVar[int] = 0

    def __init__(self, *parts):
        normalized = tuple(str(part) for part in parts)
        object.__setattr__(self, "_parts", normalized)
        type(self).hash_computations += 1
        object.__setattr__(self, "_hash", hash(normalized))

    @property
    def parts(self):
        return self._parts

    def __setattr__(self, name, value):
        raise AttributeError(f"{type(self).__name__} is immutable")

    def __eq__(self, other):
        if isinstance(other, ImmutablePathKey):
            return self._parts == other._parts
        return NotImplemented

    def __hash__(self):
        return self._hash

    def __repr__(self):
        return f"ImmutablePathKey{self._parts!r}"


ImmutablePathKey.hash_computations = 0
path_key = ImmutablePathKey("users", 42, "settings")
values = [hash(path_key) for _ in range(5)]

print(values)
print("hash computations during construction:", ImmutablePathKey.hash_computations)
show_exception("attempted mutation", lambda: setattr(path_key, "_parts", ("x",)))

[1562394987484373015, 1562394987484373015, 1562394987484373015, 1562394987484373015, 1562394987484373015]
hash computations during construction: 1
attempted mutation: AttributeError: ImmutablePathKey is immutable


### Solution takeaway

Hash caching is an optimization for an already immutable value object. It is not a substitute for immutability.

---

# Problem 12 — Measure collision cost with equality-call instrumentation

Collisions are legal. The dictionary resolves them by comparing candidate keys for equality. We can make that work visible by counting `__eq__` calls.

### Step 1: Create keys with selectable hash quality

In [38]:
class CountingKey:
    eq_calls = 0

    def __init__(self, value, *, constant_hash=False):
        self.value = value
        self.constant_hash = constant_hash

    def __eq__(self, other):
        type(self).eq_calls += 1
        if isinstance(other, CountingKey):
            return (
                self.value == other.value
                and self.constant_hash == other.constant_hash
            )
        return NotImplemented

    def __hash__(self):
        if self.constant_hash:
            return 1
        return hash(self.value)


def build_table(size, *, constant_hash):
    return {
        CountingKey(i, constant_hash=constant_hash): i
        for i in range(size)
    }

### Step 2: Compare lookup work

In [39]:
SIZE = 1_000
normal_table = build_table(SIZE, constant_hash=False)
collision_table = build_table(SIZE, constant_hash=True)

CountingKey.eq_calls = 0
print(normal_table[CountingKey(SIZE - 1, constant_hash=False)])
normal_comparisons = CountingKey.eq_calls

CountingKey.eq_calls = 0
print(collision_table[CountingKey(SIZE - 1, constant_hash=True)])
collision_comparisons = CountingKey.eq_calls

print("normal comparisons:", normal_comparisons)
print("collision comparisons:", collision_comparisons)

999
999
normal comparisons: 1
collision comparisons: 1000


With useful hashes, the dictionary usually reaches the correct candidate directly. With a constant hash, many keys share the same collision chain, so equality must be checked repeatedly.

### Step 3: Compare timing without relying on a fixed machine-specific number

In [40]:
normal_time = timeit(
    lambda: normal_table[CountingKey(SIZE - 1, constant_hash=False)],
    number=500,
)
collision_time = timeit(
    lambda: collision_table[CountingKey(SIZE - 1, constant_hash=True)],
    number=500,
)

print(f"normal lookup time:    {normal_time:.6f} seconds")
print(f"collision lookup time: {collision_time:.6f} seconds")
print(f"slowdown factor:       {collision_time / normal_time:.1f}x")

normal lookup time:    0.001563 seconds
collision lookup time: 0.312955 seconds
slowdown factor:       200.2x


### Solution takeaway

A hash function does not need to be unique, but it should distribute realistic keys well. Correctness survives collisions; performance may not.

---

# Problem 13 — Distinguish a runtime hash from a persistent identifier

Python's `hash()` is designed for in-process hash tables. It is not a portable, stable fingerprint for files, databases, or network protocols.

### Step 1: Compare string hashes under two explicit hash seeds

In [41]:
def string_hash_with_seed(text, seed):
    program = f"print(hash({text!r}))"
    env = os.environ.copy()
    env["PYTHONHASHSEED"] = str(seed)
    output = subprocess.check_output(
        [sys.executable, "-c", program],
        env=env,
        text=True,
    )
    return int(output.strip())


seed_1_hash = string_hash_with_seed("customer:42", 1)
seed_2_hash = string_hash_with_seed("customer:42", 2)

print(seed_1_hash)
print(seed_2_hash)
print(seed_1_hash == seed_2_hash)

-681652150076155593
2607877256469055809
False


Different interpreter hash seeds can produce different string hashes. This is expected and useful for security.

### Step 2: Give the value object both a runtime hash and a stable digest

In [42]:
@dataclass(frozen=True)
class CustomerCacheKey:
    region: str
    customer_id: int

    def stable_digest(self):
        payload = json.dumps(
            {
                "region": self.region,
                "customer_id": self.customer_id,
            },
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
        return sha256(payload).hexdigest()


key = CustomerCacheKey("eu", 42)
print("runtime hash:", hash(key))
print("stable digest:", key.stable_digest())

runtime hash: -6498478676980329067
stable digest: 1aecc9ea8d2b4c5206f0dae4fe15cd85423ef32bfe5579513dd482696b2f8e26


### Step 3: Verify that canonical serialization controls the digest

In [43]:
key_1 = CustomerCacheKey(region="eu", customer_id=42)
key_2 = CustomerCacheKey(customer_id=42, region="eu")

print(key_1 == key_2)
print(key_1.stable_digest() == key_2.stable_digest())

True
True


### Solution takeaway

Use `hash()` for dictionaries and sets. Use an explicitly specified serialization plus a cryptographic digest when a value must remain stable across processes, machines, or time.

---

# Problem 14 — Build a recursive freezer for memoization keys

Function-call arguments may contain lists, dictionaries, and sets. These structures are unhashable, and dictionary insertion order should not change the logical cache key.

We will convert supported containers into immutable canonical forms.

### Step 1: Define the conversion rules

In [44]:
def freeze(value):
    """Convert common nested containers into deterministic hashable values."""
    if isinstance(value, Mapping):
        frozen_items = ((freeze(k), freeze(v)) for k, v in value.items())
        return tuple(sorted(frozen_items, key=repr))

    if isinstance(value, (set, frozenset, AbstractSet)):
        return frozenset(freeze(item) for item in value)

    if isinstance(value, (list, tuple)):
        return tuple(freeze(item) for item in value)

    if isinstance(value, Hashable):
        return value

    raise TypeError(f"unsupported unhashable value: {type(value).__name__}")

### Step 2: Canonicalize differently ordered but equivalent data

In [45]:
request_1 = {
    "filters": {"active": True, "roles": ["admin", "editor"]},
    "fields": {"name", "email"},
}
request_2 = {
    "fields": {"email", "name"},
    "filters": {"roles": ["admin", "editor"], "active": True},
}

frozen_1 = freeze(request_1)
frozen_2 = freeze(request_2)

print(frozen_1)
print(frozen_1 == frozen_2)
print(hash(frozen_1) == hash(frozen_2))

(('fields', frozenset({'name', 'email'})), ('filters', (('active', True), ('roles', ('admin', 'editor')))))
True
True


### Step 3: Use the frozen structure in a call key

In [46]:
@dataclass(frozen=True)
class CallKey:
    function_name: str
    arguments: tuple
    keyword_arguments: tuple

    @classmethod
    def build(cls, function_name, args, kwargs):
        return cls(
            function_name=function_name,
            arguments=freeze(args),
            keyword_arguments=freeze(kwargs),
        )


call_1 = CallKey.build(
    "search_users",
    (),
    {"filters": request_1, "limit": 20},
)
call_2 = CallKey.build(
    "search_users",
    (),
    {"limit": 20, "filters": request_2},
)

print(call_1 == call_2)
print(hash(call_1) == hash(call_2))
print({call_1: "cached result"}[call_2])

True
True
cached result


### Step 4: State the boundary of the solution

This freezer is suitable only when its conversion rules match the application's semantics. For example, it preserves list order but ignores set order. It also sorts mapping entries by `repr`, which is practical for many controlled keys but is not a universal serialization standard.

### Solution takeaway

Canonicalization must reflect domain meaning. A generic-looking conversion function is still a policy decision and should be tested with the exact argument types the application accepts.

---

# Problem 15 — Use snapshot keys for mutable domain objects

An order is intentionally mutable: its status, items, and revision can change. Using the order object itself as a key would tie dictionary membership to mutable business state.

Instead, create a separate immutable key that identifies a particular revision.

### Step 1: Define the mutable domain object

In [47]:
class Order:
    def __init__(self, order_id, items):
        self.order_id = order_id
        self.items = list(items)
        self.status = "draft"
        self.revision = 1

    def add_item(self, item):
        self.items.append(item)
        self.revision += 1

    def submit(self):
        self.status = "submitted"
        self.revision += 1

    def __repr__(self):
        return (
            f"Order(order_id={self.order_id!r}, "
            f"revision={self.revision}, status={self.status!r})"
        )

### Step 2: Define an immutable revision key

In [48]:
@dataclass(frozen=True)
class OrderRevisionKey:
    order_id: str
    revision: int

    @classmethod
    def from_order(cls, order):
        return cls(order.order_id, order.revision)


order = Order("ORD-100", ["book"])
revision_1 = OrderRevisionKey.from_order(order)
cache = {revision_1: "price calculated at revision 1"}

print(order)
print(revision_1)

Order(order_id='ORD-100', revision=1, status='draft')
OrderRevisionKey(order_id='ORD-100', revision=1)


### Step 3: Mutate the order and create a new snapshot key

In [49]:
order.add_item("pen")
revision_2 = OrderRevisionKey.from_order(order)

print(order)
print(revision_1)
print(revision_2)
print(revision_1 == revision_2)
print(cache[revision_1])
print(cache.get(revision_2, "not calculated yet"))

Order(order_id='ORD-100', revision=2, status='draft')
OrderRevisionKey(order_id='ORD-100', revision=1)
OrderRevisionKey(order_id='ORD-100', revision=2)
False
price calculated at revision 1
not calculated yet


The old key remains valid because it represents a historical revision, not the mutable current state of the order.

### Solution takeaway

Do not force a mutable entity to behave like an immutable value object. Introduce a separate immutable key or snapshot whose identity is deliberately stable.

---

# Problem 16 — Write reusable equality and hash contract tests

Handwritten examples are useful, but a small validation helper can catch recurring mistakes across many custom key classes.

We will test:

- reflexivity: `a == a`,
- symmetry: `a == b` agrees with `b == a`,
- transitivity for equal triples,
- equal objects have equal hashes.

### Step 1: Implement the checker

In [50]:
def validate_hash_contract(samples):
    samples = list(samples)
    failures = []

    for item in samples:
        if not (item == item):
            failures.append(f"not reflexive: {item!r}")
        try:
            hash(item)
        except TypeError as exc:
            failures.append(f"unhashable: {item!r}: {exc}")

    for left, right in combinations(samples, 2):
        left_right = left == right
        right_left = right == left

        if left_right != right_left:
            failures.append(
                f"not symmetric: {left!r} == {right!r} is {left_right}, "
                f"reverse is {right_left}"
            )

        if left_right:
            try:
                if hash(left) != hash(right):
                    failures.append(
                        f"equal values have different hashes: {left!r}, {right!r}"
                    )
            except TypeError:
                pass

    for first in samples:
        for second in samples:
            for third in samples:
                if first == second and second == third and not first == third:
                    failures.append(
                        f"not transitive: {first!r}, {second!r}, {third!r}"
                    )

    return failures

### Step 2: Run it against a known broken class

In [51]:
bad_edges = [
    BadUndirectedEdge("A", "B"),
    BadUndirectedEdge("B", "A"),
    BadUndirectedEdge("A", "C"),
]

for failure in validate_hash_contract(bad_edges):
    print(failure)

equal values have different hashes: BadUndirectedEdge('A', 'B'), BadUndirectedEdge('B', 'A')


### Step 3: Run it against several repaired classes

In [52]:
good_sample_groups = {
    "ProductCode": [
        ProductCode("BOOK", 104),
        ProductCode("BOOK", 104),
        ProductCode("GAME", 104),
    ],
    "TagKey": [
        TagKey("Straße"),
        TagKey("STRASSE"),
        TagKey("Python"),
    ],
    "UndirectedEdge": [
        UndirectedEdge("A", "B"),
        UndirectedEdge("B", "A"),
        UndirectedEdge("A", "C"),
    ],
    "RouteKey": [
        RouteKey("SOF", "LHR"),
        RouteKey("SOF", "LHR"),
        RouteKey("SOF", "CDG"),
    ],
}

for name, samples in good_sample_groups.items():
    failures = validate_hash_contract(samples)
    print(f"{name:<20} failures={len(failures)}")
    for failure in failures:
        print("  ", failure)

ProductCode          failures=0
TagKey               failures=0
UndirectedEdge       failures=0
RouteKey             failures=0


### Step 4: Add domain-specific tests

A generic contract checker cannot know every business rule. Each key still needs examples for canonicalization, allowed cross-type comparisons, mutation resistance, serialization, and dictionary/set behavior.

In [53]:
def assert_lookup_equivalence(stored_key, equivalent_key):
    marker = object()
    table = {stored_key: marker}
    assert table[equivalent_key] is marker


assert_lookup_equivalence(
    TagKey("  DATA   SCIENCE "),
    TagKey("data science"),
)
assert_lookup_equivalence(
    UndirectedEdge("A", "B"),
    UndirectedEdge("B", "A"),
)
assert_lookup_equivalence(
    DateKey("2026-08-04"),
    date(2026, 8, 4),
)

print("domain-specific lookup tests passed")

domain-specific lookup tests passed


### Solution takeaway

Testing `a == b` is not enough. Hashable value objects should be tested through the collections they are designed to support.

---

# Final design checklist

Before making a custom class hashable, answer these questions:

1. **What exactly is the logical identity?** List the fields explicitly.
2. **Can any identity field change?** If yes, the object should normally be unhashable or replaced by an immutable key.
3. **Do equality and hashing use the same identity representation?**
4. **Does `__eq__` return `NotImplemented` for unsupported types?**
5. **Are cross-type equal objects guaranteed to have equal hashes?**
6. **Could subclassing change identity semantics?**
7. **Are nested identity values themselves hashable and stable?**
8. **Is canonicalization performed once at construction time?**
9. **Does the hash distribute realistic values instead of creating avoidable collisions?**
10. **Are you mistakenly treating `hash()` as a persistent identifier?**
11. **Have dictionary lookup and set deduplication been tested with independently created equal objects?**
12. **Have expected mutation attempts and edge cases been tested?**

# Additional solved mini-examples

The following compact examples reinforce the same rules in a few more domains.

## Mini-example A — Case-insensitive HTTP header names

In [54]:
@dataclass(frozen=True)
class HeaderName:
    canonical: str

    def __init__(self, value):
        object.__setattr__(self, "canonical", value.strip().casefold())

    def __str__(self):
        return self.canonical


headers = {HeaderName("Content-Type"): "application/json"}
print(headers[HeaderName(" content-type ")])

application/json


The canonical string is the only stored field, so generated equality and hashing cannot disagree about normalization.

## Mini-example B — A key with a deliberately ignored display label

In [55]:
@dataclass(frozen=True)
class AccountKey:
    account_id: int
    display_label: str = field(compare=False, hash=False)


primary = AccountKey(1001, "Main account")
renamed = AccountKey(1001, "Travel savings")

print(primary == renamed)
print({primary: "record"}[renamed])

True
record


Changing a descriptive label does not create a new account identity. Excluding the label from both comparison and hashing expresses that rule directly.

## Mini-example C — Reject unhashable identity at construction time

In [56]:
@dataclass(frozen=True)
class RuleKey:
    name: str
    parameters: tuple

    def __init__(self, name, parameters):
        frozen_parameters = freeze(parameters)
        hash(frozen_parameters)  # fail early with a clear construction boundary
        object.__setattr__(self, "name", name)
        object.__setattr__(self, "parameters", frozen_parameters)


rule = RuleKey("minimum_age", {"age": 18, "regions": ["EU", "UK"]})
print(rule)
print(hash(rule))

RuleKey(name='minimum_age', parameters=(('age', 18), ('regions', ('EU', 'UK'))))
-6824292276990005228


Failing or converting at construction time is preferable to discovering much later—during dictionary insertion—that an identity component is unhashable.

# Closing perspective

A good custom hash is rarely the most difficult part. The difficult part is defining a stable, coherent notion of equality.

Once the identity model is clear, the implementation usually follows a simple pattern:

```python
identity = (field_1, field_2, ...)
```

Use that same identity for equality and hashing, freeze or copy it at the boundary, and test it through real dictionary and set operations.